In [1]:
import json
from pathlib import Path
import pandas as pd

from collections import defaultdict

from app.schemas import ModelEvaluation, GradingResult, KeyConcepts

In [2]:
from sentence_transformers import SentenceTransformer, util

In [3]:
embedder = SentenceTransformer("cointegrated/rubert-tiny2")

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
dir_path = Path("../output/full_grading")

In [5]:
data: list[ModelEvaluation] = []

for model_dir in dir_path.iterdir():
    if model_dir.is_file():
        continue

    for evaluation_file in model_dir.iterdir():
        if evaluation_file.is_dir():
            continue

        content = evaluation_file.read_text(encoding="utf8")
        model_evaluation = ModelEvaluation.model_validate(json.loads(content))
        data.append(model_evaluation)

In [6]:
def calc_score(grading_result: GradingResult, key_concepts: KeyConcepts):
    global wrong
    global correct
    score = 0
    for i, concept_score in enumerate(grading_result.concepts):
        try:
            concept = key_concepts.concepts[concept_score.id - 1]
        except IndexError as e:
            raise ValueError("Wrong concept id")
        
        score += concept_score.coverage * concept.importance

    total_possible = sum(x.importance for x in key_concepts.concepts) * 2

    return score / total_possible * 100

In [7]:
def cosine_similarity(a, b):
    return float(util.cos_sim(a, b)[0][0])

In [17]:
scores = pd.DataFrame()

for item in data:
    try:
        grade_score = calc_score(item.grading_results, item.key_concepts)
    except ValueError as e:
        continue

    real_score = item.answer.quality
    
    embeddings = embedder.encode([item.question.reference_answer, item.answer.text])
    cos_sim = cosine_similarity(embeddings[0], embeddings[1])

    new_row = pd.DataFrame([{"model": item.model, "answer_type": item.answer.answer_type, "real_score": real_score, "model_score": grade_score, "cos_sim": cos_sim}])

    scores = pd.concat([scores, new_row], ignore_index=True)

In [19]:
scores["hybrid_score"] = (scores["model_score"] + scores["cos_sim"] * 100) / 2

In [20]:
scores

,model,answer_type,real_score,model_score,cos_sim,hybrid_score
0,llama3-8b,Отличный,93,100.000000,0.835392,91.769585
1,llama3-8b,Очень слабый,9,57.446809,0.422791,49.862965
2,llama3-8b,Слабый,31,18.571429,0.639563,41.263845
3,llama3-8b,Средний,58,68.085106,0.742527,71.168924
4,llama3-8b,Средний,58,50.000000,0.619964,55.998224
...,...,...,...,...,...,...
1477,vikhr-8b,Средний,58,26.543210,0.684211,47.482150
1478,vikhr-8b,Слабый,31,16.666667,0.597779,38.222297
1479,vikhr-8b,Очень слабый,3,16.666667,0.502621,33.464395
1480,vikhr-8b,Отличный,97,60.000000,0.885440,74.272012


In [28]:
for model in scores["model"].unique():
    print(model)
    corr = scores[scores.model == model]["real_score"].corr(scores[scores.model == model]["model_score"], method="spearman")
    corr_emb = scores[scores.model == model]["real_score"].corr(scores[scores.model == model]["hybrid_score"], method="spearman")
    print(f"Корреляция оценок модели с реальными: {corr}")
    print(f"Корреляция гибридных оценок с реальными: {corr_emb}")

    print("Корреляция реальной оценки и гибридной для ответов типа:")
    for answer_type in scores["answer_type"].unique():
        df_slice = scores[(scores.model == model) & (scores["answer_type"] == answer_type)]

        print(f"{answer_type}: {df_slice["real_score"].corr(scores["hybrid_score"], method="spearman")}")

llama3-8b
Корреляция оценок модели с реальными: 0.7611439933944707
Корреляция гибридных оценок с реальными: 0.8574347375508107
Корреляция реальной оценки и гибридной для ответов типа:
Отличный: 0.19915210093318111
Очень слабый: 0.30597314096488604
Слабый: 0.23419168643855237
Средний: -0.11539347267067852
Хороший: 0.20690167133379977
tlite-8b
Корреляция оценок модели с реальными: 0.8221905734557207
Корреляция гибридных оценок с реальными: 0.8933974231721518
Корреляция реальной оценки и гибридной для ответов типа:
Отличный: 0.19852479930010694
Очень слабый: 0.32606221993876855
Слабый: 0.25284085023452463
Средний: 0.05454404927658904
Хороший: 0.23855082561735916
vikhr-8b
Корреляция оценок модели с реальными: 0.7377093685444558
Корреляция гибридных оценок с реальными: 0.8399643020120325
Корреляция реальной оценки и гибридной для ответов типа:
Отличный: 0.12459344970840154
Очень слабый: 0.3820178754036749
Слабый: 0.1660457114692676
Средний: -0.03546094960727992
Хороший: 0.19057295221578238
